# Importation librairie

In [23]:
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import transforms, datasets, utils
from torch.utils.data import DataLoader, Subset, TensorDataset, Dataset
from PIL import Image
import torchvision.transforms as T
import os, random, math, itertools, time, torch, glob
from collections import OrderedDict, deque
from pathlib import Path
import torch.optim as optim
from collections import defaultdict
from torchvision.utils import save_image

# CycleGAN

Temps d'entrainement de 50 epochs : 3h (G=5.876 D=0.662)


## Composant de base

In [ ]:
class ResnetBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.b = nn.Sequential(
            nn.ReflectionPad2d(1), nn.Conv2d(c, c, 3), nn.InstanceNorm2d(c), nn.ReLU(True),
            nn.ReflectionPad2d(1), nn.Conv2d(c, c, 3), nn.InstanceNorm2d(c)
        )
    def forward(self, x): return x + self.b(x)

class ResnetGenerator(nn.Module):
    def __init__(self, in_c=3, out_c=3, ngf=64, n_blocks=9):
        super().__init__()
        model = [nn.ReflectionPad2d(3), nn.Conv2d(in_c, ngf, 7), nn.InstanceNorm2d(ngf), nn.ReLU(True)]
        # down
        c = ngf
        for _ in range(2):
            model += [nn.Conv2d(c, c*2, 3, stride=2, padding=1), nn.InstanceNorm2d(c*2), nn.ReLU(True)]
            c *= 2
        # res
        for _ in range(n_blocks): model += [ResnetBlock(c)]
        # up
        for _ in range(2):
            model += [nn.Upsample(scale_factor=2, mode='nearest'),
                      nn.Conv2d(c, c//2, 3, padding=1), nn.InstanceNorm2d(c//2), nn.ReLU(True)]
            c //= 2
        model += [nn.ReflectionPad2d(3), nn.Conv2d(c, out_c, 7), nn.Tanh()]
        self.net = nn.Sequential(*model)
    def forward(self, x): return self.net(x)

class NLayerDiscriminator(nn.Module):
    def __init__(self, in_c=3, ndf=64, n_layers=3):
        super().__init__()
        kw, pad = 4, 1
        seq = [nn.Conv2d(in_c, ndf, kw, 2, pad), nn.LeakyReLU(0.2, True)]
        c = ndf
        for i in range(1, n_layers):
            seq += [nn.Conv2d(c, c*2, kw, 2, pad), nn.InstanceNorm2d(c*2), nn.LeakyReLU(0.2, True)]
            c *= 2
        seq += [nn.Conv2d(c, c*2, kw, 1, pad), nn.InstanceNorm2d(c*2), nn.LeakyReLU(0.2, True)]
        seq += [nn.Conv2d(c*2, 1, kw, 1, pad)]  # carte PatchGAN
        self.net = nn.Sequential(*seq)
    def forward(self, x): return self.net(x)

def gan_loss(pred, real=True):
    target = torch.ones_like(pred) if real else torch.zeros_like(pred)
    return F.mse_loss(pred, target)  # LSGAN

# --- instanciation
G, Fm = ResnetGenerator(ngf=32), ResnetGenerator(ngf=32)
Dy, Dx = NLayerDiscriminator(ndf=32), NLayerDiscriminator(ndf=32)
optG = torch.optim.Adam(list(G.parameters())+list(Fm.parameters()), lr=2e-4, betas=(0.5,0.999))
optD = torch.optim.Adam(list(Dy.parameters())+list(Dx.parameters()), lr=2e-4, betas=(0.5,0.999))

lambda_cyc, lambda_id = 10.0, 2.0

# --- une itération d'entraînement (x: batch photos, y: batch anime, ∈[-1,1])
def train_step(x, y):
    # Discriminateurs
    with torch.no_grad():
        fake_y, fake_x = G(x), Fm(y)
    Dy_loss = gan_loss(Dy(y), True) + gan_loss(Dy(fake_y), False)
    Dx_loss = gan_loss(Dx(x), True) + gan_loss(Dx(fake_x), False)
    optD.zero_grad(); (Dy_loss+Dx_loss).backward(); optD.step()

    # Générateurs
    fake_y, fake_x = G(x), Fm(y)
    rec_x, rec_y = Fm(fake_y), G(fake_x)
    G_adv = gan_loss(Dy(fake_y), True) + gan_loss(Dx(fake_x), True)
    L_cyc = F.l1_loss(rec_x, x) + F.l1_loss(rec_y, y)
    L_id  = F.l1_loss(G(y), y) + F.l1_loss(Fm(x), x)
    G_loss = G_adv + lambda_cyc*L_cyc + lambda_id*L_id
    optG.zero_grad(); G_loss.backward(); optG.step()
    return {'G': G_loss.item(), 'D': (Dy_loss+Dx_loss).item()}

# --- inférence Photo→Anime
@torch.no_grad()
def stylize_photo(img_tensor_256):  # tensor [1,3,256,256] normalisé [-1,1]
    G.eval()
    out = G(img_tensor_256)
    return (out.clamp(-1,1)+1)/2  # [0,1]

## Importation des données

(Toutes ces données proviennent du site kaggle)

In [4]:
transform = transforms.Compose([
    transforms.Resize(286),
    transforms.RandomCrop(128),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

dataset_X = datasets.ImageFolder(root="Data/Images_Photo", transform=transform)
dataset_Y_full = datasets.ImageFolder(root="Data/Images_Anime", transform=transform)
indices = random.sample(range(len(dataset_Y_full)), 2000)
dataset_Y = Subset(dataset_Y_full, indices)

loader_X = DataLoader(dataset_X, batch_size=1, shuffle=True)
loader_Y = DataLoader(dataset_Y, batch_size=1, shuffle=True)

## Entrainement

- Photo : 734 Mo
- Anime : 833 Mo

ratio : 734/833=1.14 

- Photo : 1 671 éléments
- Anime : 82 977 éléments

ratio : 82977/1671=49.66

ratio pondéré : 1.14*49.66/ln(82977+1671)=3.856 > 2 il est fortement envisageable d'équilibré le jeu de données donc utilisation seul de 2000 images.

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
G.to(device); Fm.to(device); Dy.to(device); Dx.to(device)

for epoch in range(200):  # 200 époques typiques
    for (x,_), (y,_) in zip(loader_X, loader_Y):
        x, y = x.to(device), y.to(device)
        losses = train_step(x, y)
    print(f"epoch {epoch} | G={losses['G']:.3f} D={losses['D']:.3f}")
    if epoch % 10 == 0:
        torch.save(G.state_dict(), f"weights_G_{epoch}.pth")

epoch 0 | G=10.191 D=0.385
epoch 1 | G=11.095 D=0.671
epoch 2 | G=8.234 D=0.375
epoch 3 | G=8.860 D=1.178
epoch 4 | G=11.261 D=0.415
epoch 5 | G=6.937 D=0.536
epoch 6 | G=5.990 D=0.932
epoch 7 | G=9.138 D=0.332
epoch 8 | G=6.126 D=0.462
epoch 9 | G=7.834 D=0.402
epoch 10 | G=6.156 D=0.883
epoch 11 | G=4.675 D=0.442
epoch 12 | G=10.357 D=0.335
epoch 13 | G=5.904 D=1.301
epoch 14 | G=6.014 D=0.755
epoch 15 | G=5.675 D=0.619
epoch 16 | G=8.556 D=0.813
epoch 17 | G=10.962 D=0.204
epoch 18 | G=7.258 D=0.472
epoch 19 | G=8.046 D=0.165
epoch 20 | G=5.802 D=0.612
epoch 21 | G=5.576 D=0.640
epoch 22 | G=5.821 D=0.807
epoch 23 | G=7.423 D=0.404
epoch 24 | G=8.509 D=0.522
epoch 25 | G=6.183 D=0.636
epoch 26 | G=5.357 D=0.584
epoch 27 | G=6.866 D=0.231
epoch 28 | G=10.508 D=0.182
epoch 29 | G=5.670 D=0.428
epoch 30 | G=6.548 D=0.183
epoch 31 | G=4.631 D=0.792
epoch 32 | G=7.697 D=1.014
epoch 33 | G=6.110 D=0.428
epoch 34 | G=5.859 D=0.232
epoch 35 | G=6.151 D=0.538
epoch 36 | G=6.755 D=0.706
epoch

KeyboardInterrupt: 

## Test

(Photo célébre de GigaChad)

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
G.load_state_dict(torch.load("weights_G_50.pth", map_location=device))
G.to(device)

img = Image.open("test_photo.png").convert("RGB")
tfm = T.Compose([
    T.Resize(256), T.CenterCrop(256),
    T.ToTensor(), T.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
x = tfm(img).unsqueeze(0).to(device)
out = stylize_photo(x)
T.ToPILImage()(out.squeeze().cpu()).save("result_anime.jpg")

# Expérimentale Adaptative CycleGAN (AC-GAN)

<b>Les différents autres GAN présenté ici auront pour but d'améliorer la vitesse de l'entrainement tout en conservant un maximum de performance dans les modèle.</b>

On commence par une idée de CycleGAN qui va progressivement rajouter des layers au fur et a mesure de son entrainement et qui va en même temps filtrer les poids qui ne bouge plus beaucoup pour pouvoir déjà faire les premiers ajustements des poids très important lors de l'initialisation rapidement, et ensuite aller vers le détail par la suite. Cela permettra en théorie d'accélérer le processus d'entrainement tout en ayant toujours une performance relativement importante.

## Config

In [ ]:
# ---------- Config ----------
DATA_ROOT = "Data"
ANIME_DIR = f"{DATA_ROOT}/Images_Anime"
PHOTO_DIR = f"{DATA_ROOT}/Images_Photo"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 128
BATCH_SIZE = 1
LR = 2e-4
BETA1, BETA2 = 0.5, 0.999
LAMBDA_CYC = 10.0
LAMBDA_ID = 3.0

START_LAYERS = 8
ADD_EVERY_EPOCHS = 10
ADD_LAYERS = 4
MAX_LAYERS = 64
MIN_STEPS_BEFORE_FREEZE = 20000
EMA_ALPHA = 0.99
EPS_FREEZE = 1e-6
FREEZE_CHECK_EVERY = 10000  # vérifier gel des poids toutes les 500 steps
CHECKPOINT_DIR = "checkpoints_acgan"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## Composant

In [9]:

# ---------- Models (minimal progressive blocks) ----------
def conv_block(in_c, out_c, stride=1):
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False),
        nn.InstanceNorm2d(out_c),
        nn.ReLU(True)
    )

class ProgressiveGen(nn.Module):
    def __init__(self, in_c=3, out_c=3, base_c=64, max_layers=MAX_LAYERS):
        super().__init__()
        self.base_c = base_c
        self.max_layers = max_layers
        self.head = nn.Sequential(nn.ReflectionPad2d(3),
                                  nn.Conv2d(in_c, base_c, 7, 1, 0, bias=False),
                                  nn.InstanceNorm2d(base_c), nn.ReLU(True))
        self.layers = nn.ModuleList([conv_block(base_c, base_c) for _ in range(max_layers)])
        self.final = nn.Sequential(nn.Conv2d(base_c, out_c, 7, 1, 3), nn.Tanh())
    def forward(self, x, active_layers):
        x = self.head(x)
        for i in range(active_layers):
            x = self.layers[i](x)
        return self.final(x)
    def add_layers(self, n):
        # layers preallocated; active count is controlled by training loop
        pass

class ProgressiveDisc(nn.Module):
    def __init__(self, in_c=3, base_c=64, max_layers=MAX_LAYERS):
        super().__init__()
        self.base = nn.Conv2d(in_c, base_c, 4, 2, 1)
        self.layers = nn.ModuleList([nn.Sequential(
            nn.Conv2d(base_c, base_c, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(base_c),
            nn.LeakyReLU(0.2, True)
        ) for _ in range(max_layers)])
        self.final = nn.Conv2d(base_c, 1, 4, 1, 1)
    def forward(self, x, active_layers):
        x = F.leaky_relu(self.base(x), 0.2, True)
        for i in range(active_layers):
            x = self.layers[i](x)
        return self.final(x)
    def add_layers(self, n):
        pass

# ---------- Utilities: dataloaders ----------
def make_loaders(img_size=IMG_SIZE, batch_size=BATCH_SIZE):
    tf = transforms.Compose([
        transforms.Resize(int(img_size*1.125)),
        transforms.RandomCrop(img_size),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5,)*3, (0.5,)*3)
    ])
    ds_p = datasets.ImageFolder(PHOTO_DIR, transform=tf)
    ds_a = datasets.ImageFolder(ANIME_DIR, transform=tf)
    loader_p = DataLoader(ds_p, batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)
    loader_a = DataLoader(ds_a, batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)
    return loader_p, loader_a

# ---------- Adaptive per-weight state ----------
class ParamState:
    def __init__(self, p):
        self.ema = torch.zeros_like(p.data)
        self.history = deque(maxlen=200)

def build_param_states(model):
    return {name: ParamState(param.detach()) for name, param in model.named_parameters()}

def apply_freeze_masks_from_states(model, states):
    # set requires_grad according to EMA threshold
    for name, param in model.named_parameters():
        st = states[name]
        # compute scalar metric
        metric = float(st.ema.abs().mean().cpu())
        if st.ema is not None and metric < EPS_FREEZE:
            param.requires_grad = False

def update_states_after_backward(model, states):
    for name, param in model.named_parameters():
        st = states[name]
        if param.grad is None:
            continue
        g = param.grad.detach()
        st.ema.mul_(EMA_ALPHA).add_( (1 - EMA_ALPHA) * g.abs().to(st.ema.device) )
        # zero grads where param.requires_grad == False
        if not param.requires_grad:
            param.grad.detach().zero_()

# ---------- Loss ----------
def lsgan_loss(pred, target_is_real=True):
    target = torch.ones_like(pred) if target_is_real else torch.zeros_like(pred)
    return F.mse_loss(pred, target)

# ---------- Train function (single) ----------
def train_acgan(epochs=100):
    # model init
    G_AB = ProgressiveGen().to(DEVICE)   # photo -> anime
    G_BA = ProgressiveGen().to(DEVICE)   # anime -> photo
    D_A = ProgressiveDisc().to(DEVICE)   # real photo discriminator
    D_B = ProgressiveDisc().to(DEVICE)   # real anime discriminator

    optG = optim.Adam(list(G_AB.parameters()) + list(G_BA.parameters()), lr=LR, betas=(BETA1, BETA2))
    optD = optim.Adam(list(D_A.parameters()) + list(D_B.parameters()), lr=LR, betas=(BETA1, BETA2))

    states_GAB = build_param_states(G_AB)
    states_GBA = build_param_states(G_BA)
    states_DA = build_param_states(D_A)
    states_DB = build_param_states(D_B)

    loader_X, loader_Y = make_loaders()
    iter_X = iter(loader_X)
    iter_Y = iter(loader_Y)

    active_layers = START_LAYERS
    global_step = 0

    for epoch in range(1, epochs+1):
        # add layers every ADD_EVERY_EPOCHS
        if epoch % ADD_EVERY_EPOCHS == 0:
            active_layers = min(active_layers + ADD_LAYERS, MAX_LAYERS)
            print(f"Epoch {epoch}: increased active_layers -> {active_layers}")

        steps = min(len(loader_X), len(loader_Y))
        for i in range(steps):
            global_step += 1
            try:
                x, _ = next(iter_X)
            except StopIteration:
                iter_X = iter(loader_X); x, _ = next(iter_X)
            try:
                y, _ = next(iter_Y)
            except StopIteration:
                iter_Y = iter(loader_Y); y, _ = next(iter_Y)

            x = x.to(DEVICE); y = y.to(DEVICE)

            # -----------------  D update  -----------------
            with torch.no_grad():
                fake_y = G_AB(x, active_layers)
                fake_x = G_BA(y, active_layers)

            D_A_real = D_A(x, active_layers)
            D_A_fake = D_A(fake_x.detach(), active_layers)
            loss_DA = lsgan_loss(D_A_real, True) + lsgan_loss(D_A_fake, False)

            D_B_real = D_B(y, active_layers)
            D_B_fake = D_B(fake_y.detach(), active_layers)
            loss_DB = lsgan_loss(D_B_real, True) + lsgan_loss(D_B_fake, False)

            loss_D = 0.5 * (loss_DA + loss_DB)
            optD.zero_grad()
            loss_D.backward()
            if global_step >= MIN_STEPS_BEFORE_FREEZE:
                update_states_after_backward(D_A, states_DA)
                update_states_after_backward(D_B, states_DB)
                apply_freeze_masks_from_states(D_A, states_DA)
                apply_freeze_masks_from_states(D_B, states_DB)
            optD.step()

            # -----------------  G update  -----------------
            fake_y = G_AB(x, active_layers)
            fake_x = G_BA(y, active_layers)
            rec_x = G_BA(fake_y, active_layers)
            rec_y = G_AB(fake_x, active_layers)

            loss_G_adv = lsgan_loss(D_B(fake_y, active_layers), True) + lsgan_loss(D_A(fake_x, active_layers), True)
            loss_cyc = F.l1_loss(rec_x, x) + F.l1_loss(rec_y, y)
            loss_id = F.l1_loss(G_AB(y, active_layers), y) + F.l1_loss(G_BA(x, active_layers), x)

            loss_G = loss_G_adv + LAMBDA_CYC * loss_cyc + LAMBDA_ID * loss_id

            optG.zero_grad()
            loss_G.backward()
            if global_step >= MIN_STEPS_BEFORE_FREEZE:
                update_states_after_backward(G_AB, states_GAB)
                update_states_after_backward(G_BA, states_GBA)
                apply_freeze_masks_from_states(G_AB, states_GAB)
                apply_freeze_masks_from_states(G_BA, states_GBA)
            optG.step()

            if global_step % 100 == 0:
                print(f"epoch {epoch} step {global_step} numlayers {active_layers}| G={loss_G.item():.4f} D={loss_D.item():.4f}")

        # checkpoint end of epoch
        torch.save({
            "epoch": epoch,
            "G_AB": G_AB.state_dict(),
            "G_BA": G_BA.state_dict(),
            "D_A": D_A.state_dict(),
            "D_B": D_B.state_dict(),
            "optG": optG.state_dict(),
            "optD": optD.state_dict(),
            "active_layers": active_layers
        }, f"{CHECKPOINT_DIR}/acgan_epoch{epoch}.pth")
        print(f"Saved checkpoint epoch {epoch}")

    print("Training finished.")

## Entrainement

In [10]:
if __name__ == "__main__":
    train_acgan(epochs=60)

epoch 1 step 100 numlayers 4| G=6.6057 D=0.6408
epoch 1 step 200 numlayers 4| G=8.1347 D=0.5939
epoch 1 step 300 numlayers 4| G=9.3784 D=0.4417
epoch 1 step 400 numlayers 4| G=10.2395 D=0.4767
epoch 1 step 500 numlayers 4| G=7.4012 D=0.3325
epoch 1 step 600 numlayers 4| G=7.0964 D=0.3811
epoch 1 step 700 numlayers 4| G=8.3282 D=0.3043
epoch 1 step 800 numlayers 4| G=9.8012 D=0.2459
epoch 1 step 900 numlayers 4| G=8.2121 D=0.4950
epoch 1 step 1000 numlayers 4| G=6.4364 D=0.3320
epoch 1 step 1100 numlayers 4| G=5.1560 D=0.7122
epoch 1 step 1200 numlayers 4| G=6.6110 D=0.2188
epoch 1 step 1300 numlayers 4| G=6.4642 D=0.5663
epoch 1 step 1400 numlayers 4| G=10.3268 D=0.6566
epoch 1 step 1500 numlayers 4| G=6.8393 D=0.2700
epoch 1 step 1600 numlayers 4| G=6.9046 D=0.1789
Saved checkpoint epoch 1
epoch 2 step 1700 numlayers 4| G=6.1089 D=0.2344
epoch 2 step 1800 numlayers 4| G=6.2908 D=0.2286
epoch 2 step 1900 numlayers 4| G=7.3484 D=0.6123
epoch 2 step 2000 numlayers 4| G=6.8994 D=0.1748
ep

ValueError: Expected more than 1 spatial element when training, got input size torch.Size([1, 64, 1, 1])

## Test

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
G.load_state_dict(torch.load("checkpoints_acgan/acgan_epoch7.pth", map_location=device))
G.to(device)

img = Image.open("test_photo.png").convert("RGB")
tfm = T.Compose([
    T.Resize(256), T.CenterCrop(256),
    T.ToTensor(), T.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
x = tfm(img).unsqueeze(0).to(device)
out = stylize_photo(x)
T.ToPILImage()(out.squeeze().cpu()).save("result_anime.jpg")

NameError: name 'G' is not defined

## V2

Meilleur performance pour moins d'étape observer
7 epochs, 34 min mais 36 layers, Loss G=0.2470 D=0.2467

In [ ]:
# ---------------------------
# Définition du modèle maximal
# ---------------------------
class Generator(nn.Module):
    def __init__(self, max_layers=32, layer_channels=64):
        super().__init__()
        self.max_layers = max_layers
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(layer_channels, layer_channels, 3, padding=1),
                nn.ReLU()
            ) for _ in range(max_layers)
        ])
        self.initial = nn.Conv2d(3, layer_channels, 3, padding=1)
        self.final = nn.Conv2d(layer_channels, 3, 3, padding=1)
        self.active_layers = 4  # début avec 4 couches

    def forward(self, x):
        x = self.initial(x)
        for i in range(self.active_layers):
            x = self.layers[i](x)
        x = self.final(x)
        return x

# Même principe pour le Discriminateur
class Discriminator(nn.Module):
    def __init__(self, max_layers=32, layer_channels=64):
        super().__init__()
        self.max_layers = max_layers
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(3 if i==0 else layer_channels, layer_channels, 3, padding=1),
                nn.LeakyReLU(0.2)
            ) for i in range(max_layers)
        ])
        self.final = nn.Conv2d(layer_channels, 1, 3, padding=1)
        self.active_layers = 4

    def forward(self, x):
        for i in range(self.active_layers):
            x = self.layers[i](x)
        x = self.final(x)
        return x

# ---------------------------
# Fonction utilitaire freeze %
# ---------------------------
def freeze_stats(mask_dict):
    total = 0
    frozen = 0
    for mask in mask_dict.values():
        total += mask.numel()
        frozen += mask.sum().item()
    return 100 * frozen / total

# ---------------------------
# Training
# ---------------------------
def train_acgan(generator, discriminator, dataloader):
    device = DEVICE
    generator.to(device)
    discriminator.to(device)

    optimizer_G = optim.Adam(generator.parameters(), lr=LR, betas=(BETA1, BETA2))
    optimizer_D = optim.Adam(discriminator.parameters(), lr=LR, betas=(BETA1, BETA2))

    # dictionnaires pour historique des poids et masques
    weight_hist = {name: deque(maxlen=MIN_STEPS_BEFORE_FREEZE) for name, p in generator.named_parameters()}
    prev_param = {name: p.data.clone() for name, p in generator.named_parameters()}
    mask_dict = {name: torch.zeros_like(p, dtype=torch.bool) for name, p in generator.named_parameters()}

    criterion = nn.MSELoss()

    global_step = 0
    generator.active_layers = START_LAYERS
    discriminator.active_layers = START_LAYERS

    for epoch in range(0, 50):
        # augmentation progressive des layers
        if epoch % ADD_EVERY_EPOCHS == 0 and epoch > 0:
            generator.active_layers = min(generator.active_layers + ADD_LAYERS, MAX_LAYERS)
            discriminator.active_layers = min(discriminator.active_layers + ADD_LAYERS, MAX_LAYERS)

        for real_photos, real_anime in dataloader:
            real_photos = real_photos.to(device)
            real_anime = real_anime.to(device)

            # -----------------
            # Train Discriminator
            # -----------------
            optimizer_D.zero_grad()
            fake_anime = generator(real_photos)
            loss_D_real = criterion(discriminator(real_anime), torch.ones_like(discriminator(real_anime)))
            loss_D_fake = criterion(discriminator(fake_anime.detach()), torch.zeros_like(discriminator(fake_anime)))
            loss_D = (loss_D_real + loss_D_fake) * 0.5
            loss_D.backward()
            optimizer_D.step()

            # -----------------
            # Train Generator
            # -----------------
            optimizer_G.zero_grad()
            fake_anime = generator(real_photos)
            loss_G = criterion(discriminator(fake_anime), torch.ones_like(discriminator(fake_anime)))
            loss_G.backward()

            # -----------------
            # Update poids avec historique et gel
            # -----------------
            if global_step % FREEZE_CHECK_EVERY == 0 and global_step > 20000:
                with torch.no_grad():
                    for name, param in generator.named_parameters():
                        if param.requires_grad:
                            # Calculer le changement relatif absolu
                            delta = (param.data - prev_param[name]).abs()  # tensor des changements absolus
                            weight_hist[name].append(delta.clone())

                            # Somme des changements absolus sur l'historique
                            if len(weight_hist[name]) == 200:
                                total_abs_change = torch.stack(list(weight_hist[name])).sum()
                                if total_abs_change < EPS_FREEZE:
                                    mask_dict[name][:] = True
                                    param.requires_grad = False
                                    del weight_hist[name]  # supprimer historique pour libérer mémoire

                            prev_param[name] = param.data.clone()

                            # Appliquer le mask pour bloquer les gradients
                            if mask_dict[name].any():
                                if param.grad is not None:
                                    param.grad[mask_dict[name]] = 0

            optimizer_G.step()
            global_step += 1

        # -----------------
        # Sauvegarde du modèle
        # -----------------
        torch.save({
            'epoch': epoch,
            'generator_state_dict': generator.state_dict(),
            'discriminator_state_dict': discriminator.state_dict(),
            'optimizer_G_state_dict': optimizer_G.state_dict(),
            'optimizer_D_state_dict': optimizer_D.state_dict(),
            'mask_dict': mask_dict
        }, os.path.join(CHECKPOINT_DIR, f"acgan_epoch_{epoch}.pt"))

        print(f"Epoch {epoch} | Global step {global_step} | Active layers {generator.active_layers} | Loss G={loss_G.item():.4f} D={loss_D.item():.4f}")


## Entrainement

In [4]:
# ---------- Transforms ----------
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)  # [-1,1]
])

# ---------- Datasets ----------
anime_dataset = datasets.ImageFolder(ANIME_DIR, transform=transform)
photo_dataset = datasets.ImageFolder(PHOTO_DIR, transform=transform)

# ---------- DataLoader ----------
# Fusionne les deux datasets dans un seul pour simplifier la boucle
class PairedDataset(torch.utils.data.Dataset):
    def __init__(self, photo_dataset, anime_dataset):
        self.photo_dataset = photo_dataset
        self.anime_dataset = anime_dataset
        self.min_len = min(len(photo_dataset), len(anime_dataset))
    def __len__(self):
        return self.min_len
    def __getitem__(self, idx):
        photo_img, _ = self.photo_dataset[idx]
        anime_img, _ = self.anime_dataset[idx]
        return photo_img, anime_img

paired_dataset = PairedDataset(photo_dataset, anime_dataset)
dataloader = DataLoader(paired_dataset, batch_size=BATCH_SIZE, shuffle=True)

# ---------- Instanciation des modèles ----------
generator = Generator(max_layers=MAX_LAYERS, layer_channels=64)
discriminator = Discriminator(max_layers=MAX_LAYERS, layer_channels=64)

# ---------- Lancement de l'entraînement ----------
train_acgan(generator, discriminator, dataloader)

Epoch 0 | Global step 1668 | Active layers 8 | Loss G=0.2552 D=0.2276
Epoch 1 | Global step 3336 | Active layers 8 | Loss G=0.2406 D=0.2480
Epoch 2 | Global step 5004 | Active layers 8 | Loss G=0.3158 D=0.2534
Epoch 3 | Global step 6672 | Active layers 8 | Loss G=0.3926 D=0.2507
Epoch 4 | Global step 8340 | Active layers 8 | Loss G=0.2381 D=0.2398
Epoch 5 | Global step 10008 | Active layers 8 | Loss G=0.2779 D=0.2375
Epoch 6 | Global step 11676 | Active layers 8 | Loss G=0.2530 D=0.2710
Epoch 7 | Global step 13344 | Active layers 8 | Loss G=0.2367 D=0.2413
Epoch 8 | Global step 15012 | Active layers 8 | Loss G=0.2320 D=0.2441
Epoch 9 | Global step 16680 | Active layers 8 | Loss G=0.2826 D=0.2505
Epoch 10 | Global step 18348 | Active layers 12 | Loss G=0.3368 D=0.2494
Epoch 11 | Global step 20016 | Active layers 12 | Loss G=0.2811 D=0.1755
Epoch 12 | Global step 21684 | Active layers 12 | Loss G=0.2717 D=0.2434
Epoch 13 | Global step 23352 | Active layers 12 | Loss G=0.2718 D=0.2501
Epo

## Test

In [5]:
def stylize_image(G, checkpoint_path, image_path, output_path="result_anime.jpg", device=None, img_size=128):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    # charger checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
    G.load_state_dict(checkpoint['generator_state_dict'])
    G.to(device)
    G.eval()

    # déterminer automatiquement le nombre de layers du checkpoint si défini
    if hasattr(checkpoint, 'active_layers'):
        G.active_layers = checkpoint['active_layers']
    else:
        # sinon on active toutes les couches par défaut
        G.active_layers = getattr(G, 'max_layers', 64)

    # charger et transformer l'image
    img = Image.open(image_path).convert("RGB")
    tfm = T.Compose([
        T.Resize(img_size),
        T.CenterCrop(img_size),
        T.ToTensor(),
        T.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
    ])
    x = tfm(img).unsqueeze(0).to(device)

    # générer l'image stylisée
    with torch.no_grad():
        out = G(x)

    # dé-normaliser et sauvegarder
    result_img = T.ToPILImage()(0.5 * (out.squeeze().cpu() + 1.0))
    result_img.save(output_path)
    print(f"Stylisation terminée ! Résultat sauvegardé dans {output_path}")

stylize_image(G=generator,
    checkpoint_path="checkpoints_acgan/acgan_epoch_49.pt",
    image_path="test_photo.png",
    output_path="result_anime.jpg",
    img_size=128
)

Stylisation terminée ! Résultat sauvegardé dans result_anime.jpg


### Test non concluant, a peaufiner
### Bon au final je vais passez à une autre structure plus prometteuse

# Expérimentale Graphe CycleGAN (GC-GAN) (en cours de création)

Ensuite ici on va expérimenter un modèle de CycleGAN en graphe qui sera théoriquement plus modulaire et nécessitant moins de poids (topologiquement parlant c'est une optimisation du réseau).

## V1

In [17]:
# ---------- Graph Node (bloc de base) ----------
class GraphNode(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)

# ---------- Graph Generator (séquentiel) ----------
class GraphGenerator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, num_blocks=4):
        super().__init__()
        layers = []
        prev_channels = in_channels
        for _ in range(num_blocks):
            layers.append(GraphNode(prev_channels, 64))
            prev_channels = 64
        self.blocks = nn.Sequential(*layers)
        self.final_conv = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        out = self.blocks(x)
        return torch.tanh(self.final_conv(out))

# ---------- Graph Discriminator (simple séquentiel aussi) ----------
class GraphDiscriminator(nn.Module):
    def __init__(self, in_channels=3, num_blocks=3):
        super().__init__()
        layers = []
        prev_channels = in_channels
        for _ in range(num_blocks):
            layers.append(GraphNode(prev_channels, 64))
            prev_channels = 64
        self.blocks = nn.Sequential(*layers)
        self.final_conv = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        out = self.blocks(x)
        return self.final_conv(out)

# ---------- CycleGAN Loss ----------
class CycleGANLoss:
    def __init__(self, device='cuda'):
        self.adv_loss = nn.MSELoss()
        self.cycle_loss = nn.L1Loss()
        self.id_loss = nn.L1Loss()
        self.device = device

    def generator_loss(self, fake, disc_fake, real, identity):
        adv = self.adv_loss(disc_fake, torch.ones_like(disc_fake))
        cycle = self.cycle_loss(fake, real)
        idt = self.id_loss(identity, real)
        return adv + 10*cycle + 3*idt

    def discriminator_loss(self, disc_real, disc_fake):
        loss_real = self.adv_loss(disc_real, torch.ones_like(disc_real))
        loss_fake = self.adv_loss(disc_fake, torch.zeros_like(disc_fake))
        return 0.5 * (loss_real + loss_fake)

## Entrainement

In [ ]:
# ---------- Dataset custom ----------
class SimpleImageDataset(Dataset):
    def __init__(self, folder, transform=None):
        self.paths = glob.glob(f"{folder}/*.*")
        self.transform = transform
        if len(self.paths) == 0:
            raise ValueError(f"Aucune image trouvée dans {folder}")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

# ---------- Transform ----------
def get_transform(img_size=128):
    return transforms.Compose([
        transforms.Resize(img_size),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
    ])

# ---------- Entraînement ----------
def train_cyclegan(data_root="Data", epochs=10, batch_size=1, img_size=128, lr=2e-4, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    PHOTO_DIR = os.path.join(data_root, "Images_Photo")
    ANIME_DIR = os.path.join(data_root, "Images_Anime")

    tfm = get_transform(img_size)
    loader_X = DataLoader(SimpleImageDataset(PHOTO_DIR, transform=tfm),
                          batch_size=batch_size, shuffle=True)
    loader_Y = DataLoader(SimpleImageDataset(ANIME_DIR, transform=tfm),
                          batch_size=batch_size, shuffle=True)

    # Modèles
    G_XtoY = GraphGenerator().to(device)
    G_YtoX = GraphGenerator().to(device)
    D_X = GraphDiscriminator().to(device)
    D_Y = GraphDiscriminator().to(device)

    # Optimizers
    opt_G = optim.Adam(list(G_XtoY.parameters()) + list(G_YtoX.parameters()), lr=lr, betas=(0.5,0.999))
    opt_D_X = optim.Adam(D_X.parameters(), lr=lr, betas=(0.5,0.999))
    opt_D_Y = optim.Adam(D_Y.parameters(), lr=lr, betas=(0.5,0.999))

    # Loss
    criterion = CycleGANLoss(device=device)

    os.makedirs("GraphCycleGAN_checkpoints", exist_ok=True)

    for epoch in range(epochs):
        for real_X, real_Y in zip(loader_X, loader_Y):
            real_X = real_X.to(device)
            real_Y = real_Y.to(device)

            # ----- Générateurs -----
            fake_Y = G_XtoY(real_X)
            fake_X = G_YtoX(real_Y)

            rec_X = G_YtoX(fake_Y)
            rec_Y = G_XtoY(fake_X)

            idt_X = G_YtoX(real_X)
            idt_Y = G_XtoY(real_Y)

            disc_fake_Y = D_Y(fake_Y)
            disc_fake_X = D_X(fake_X)

            g_loss = (
                criterion.generator_loss(fake_Y, disc_fake_Y, real_X, idt_Y) +
                criterion.generator_loss(fake_X, disc_fake_X, real_Y, idt_X)
            )

            opt_G.zero_grad()
            g_loss.backward()
            opt_G.step()

            # ----- Discriminateurs -----
            disc_real_X = D_X(real_X)
            disc_real_Y = D_Y(real_Y)

            disc_fake_X_det = D_X(fake_X.detach())
            disc_fake_Y_det = D_Y(fake_Y.detach())

            d_x_loss = criterion.discriminator_loss(disc_real_X, disc_fake_X_det)
            d_y_loss = criterion.discriminator_loss(disc_real_Y, disc_fake_Y_det)

            opt_D_X.zero_grad()
            d_x_loss.backward()
            opt_D_X.step()

            opt_D_Y.zero_grad()
            d_y_loss.backward()
            opt_D_Y.step()

        print(f"Epoch {epoch+1}/{epochs} | G_loss: {g_loss.item():.4f} | D_X: {d_x_loss.item():.4f} | D_Y: {d_y_loss.item():.4f}")

        # Sauvegarde d'exemples
        save_image((fake_Y*0.5+0.5), f"fakeY_epoch{epoch+1}.png")
        save_image((fake_X*0.5+0.5), f"fakeX_epoch{epoch+1}.png")

        # Sauvegarde checkpoint
        torch.save({
            "G_XtoY": G_XtoY.state_dict(),
            "G_YtoX": G_YtoX.state_dict(),
            "D_X": D_X.state_dict(),
            "D_Y": D_Y.state_dict(),
            "opt_G": opt_G.state_dict(),
            "opt_D_X": opt_D_X.state_dict(),
            "opt_D_Y": opt_D_Y.state_dict()
        }, f"GraphCycleGAN_checkpoints/cyclegan_epoch{epoch+1}.pt")

    print("Entraînement terminé.")

train_cyclegan(data_root="Data", epochs=10, batch_size=1, img_size=128)

ValueError: Aucune image trouvée dans Data/Images_Photo/*

## Test

In [ ]:
def stylize_with_cyclegan(G, checkpoint_path=None, image_path="test_photo.png",
                          output_path="result_anime.jpg", device=None, img_size=128):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    if checkpoint_path and os.path.isfile(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)
        G.load_state_dict(checkpoint['G_XtoY'])
        print("Checkpoint chargé :", checkpoint_path)
    else:
        print("Aucun checkpoint trouvé, exécution avec poids aléatoires.")

    G.to(device).eval()

    img = Image.open(image_path).convert("RGB")
    tfm = T.Compose([
        T.Resize(img_size),
        T.CenterCrop(img_size),
        T.ToTensor(),
        T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    x = tfm(img).unsqueeze(0).to(device)

    with torch.no_grad():
        out = G(x)

    out = 0.5 * (out.squeeze().cpu() + 1.0)
    result_img = T.ToPILImage()(out.clamp(0,1))
    result_img.save(output_path)
    print(f"Résultat sauvegardé dans {output_path}")

# ---------- Exemple d'utilisation ----------
stylize_with_cyclegan(
    G=GraphGenerator(),
    image_path="test_photo.png",
    output_path="result_anime.jpg",
    img_size=128
)

# Expérimentale Adaptative Graphe CycleGAN (AGC-GAN)

Et ici, on va expérimenter une fusion des 2 concepts pour ainsi avoir un modèle qui garde un maximum de performance pour un temps de calcule moindre.